# CENG 467 — Question 4: Machine Translation

**Author:** Yusuf Furkan Aktay  
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Instructor:** Prof. Dr. Aytuğ Onan

## Objective
Compare a custom Seq2Seq model with Bahdanau attention against a pre-trained Transformer (Helsinki-NLP/opus-mt-en-de) on the Multi30k English→German translation task. Evaluate using BLEU, METEOR, ChrF, and BERTScore.

## Setup
- **Runtime:** Google Colab — GPU (T4)
- **Random Seed:** 42
- **Dataset:** Multi30k (EN→DE)
- **Models:** Seq2Seq+Attention (custom, trained from scratch), opus-mt-en-de (pre-trained, inference only)

## 1. Environment Setup

In [1]:
!pip install -q --upgrade transformers accelerate datasets sentencepiece
!pip install -q sacrebleu nltk bert-score

In [2]:
import os, random, json, time, warnings, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

os.makedirs('results', exist_ok=True)

Device: cuda


## 2. Dataset: Multi30k (EN→DE)

Multi30k is a multilingual extension of Flickr30k, where each English image caption is paired with a German translation. We download the canonical splits directly from the official Multi30k GitHub repository to avoid HuggingFace API issues.
Splits: 29,000 train / 1,014 valid / 1,000 test.

In [3]:
from datasets import load_dataset

print('Downloading Multi30k from HuggingFace mirror...')
hf_ds = load_dataset('bentrevett/multi30k')

# Convert to the same dict-of-lists structure as before so the rest of the notebook works unchanged
data = {
    'train_en': [x['en'] for x in hf_ds['train']],
    'train_de': [x['de'] for x in hf_ds['train']],
    'val_en':   [x['en'] for x in hf_ds['validation']],
    'val_de':   [x['de'] for x in hf_ds['validation']],
    'test_en':  [x['en'] for x in hf_ds['test']],
    'test_de':  [x['de'] for x in hf_ds['test']],
}

print(f'Train: {len(data["train_en"])} | Val: {len(data["val_en"])} | Test: {len(data["test_en"])}')
print('\n--- Sample ---')
print('EN:', data['train_en'][0])
print('DE:', data['train_de'][0])

README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 29000 | Val: 1014 | Test: 1000

--- Sample ---
EN: Two young, White males are outside near many bushes.
DE: Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


## 3. Preprocessing & Vocab (for Seq2Seq)

Whitespace tokenization, lowercasing, and frequency-based vocabulary construction (min_freq=2).

In [4]:
from collections import Counter
import re

PAD, SOS, EOS, UNK = 0, 1, 2, 3
MAX_LEN = 30

def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    return text.split()

def build_vocab(sentences, min_freq=2, max_size=10000):
    counter = Counter()
    for s in sentences: counter.update(tokenize(s))
    vocab = {'<pad>':PAD, '<sos>':SOS, '<eos>':EOS, '<unk>':UNK}
    for w, c in counter.most_common(max_size):
        if c >= min_freq: vocab[w] = len(vocab)
    return vocab

src_vocab = build_vocab(data['train_en'])
tgt_vocab = build_vocab(data['train_de'])
src_inv = {i:w for w,i in src_vocab.items()}
tgt_inv = {i:w for w,i in tgt_vocab.items()}
print(f'EN vocab: {len(src_vocab)} | DE vocab: {len(tgt_vocab)}')

def encode(text, vocab, add_sos_eos=True):
    ids = [vocab.get(t, UNK) for t in tokenize(text)][:MAX_LEN-2]
    if add_sos_eos: ids = [SOS] + ids + [EOS]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids[:MAX_LEN]

class MTDataset(Dataset):
    def __init__(self, src_lines, tgt_lines):
        self.src = torch.tensor([encode(s, src_vocab) for s in src_lines], dtype=torch.long)
        self.tgt = torch.tensor([encode(s, tgt_vocab) for s in tgt_lines], dtype=torch.long)
    def __len__(self): return len(self.src)
    def __getitem__(self, i): return self.src[i], self.tgt[i]

train_ds = MTDataset(data['train_en'], data['train_de'])
test_ds  = MTDataset(data['test_en'],  data['test_de'])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=64)

EN vocab: 5970 | DE vocab: 7811


## 4. Model 1 — Seq2Seq with Bahdanau Attention

A bidirectional GRU encoder, additive (Bahdanau) attention, and a unidirectional GRU decoder.

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hid=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hid, batch_first=True, bidirectional=True)
        self.fc  = nn.Linear(hid*2, hid)
    def forward(self, x):
        e = self.emb(x)
        out, h = self.gru(e)            # out: [B,T,2H]; h: [2,B,H]
        h = torch.tanh(self.fc(torch.cat([h[0], h[1]], dim=1))).unsqueeze(0)  # [1,B,H]
        return out, h

class BahdanauAttention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.W = nn.Linear(hid, hid)
        self.U = nn.Linear(hid*2, hid)
        self.v = nn.Linear(hid, 1, bias=False)
    def forward(self, dec_h, enc_out, mask):
        # dec_h: [B,H]; enc_out: [B,T,2H]
        scores = self.v(torch.tanh(self.W(dec_h).unsqueeze(1) + self.U(enc_out))).squeeze(-1)  # [B,T]
        scores = scores.masked_fill(~mask, -1e9)
        attn = F.softmax(scores, dim=-1)
        ctx = torch.bmm(attn.unsqueeze(1), enc_out).squeeze(1)  # [B,2H]
        return ctx, attn

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hid=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.attn = BahdanauAttention(hid)
        self.gru = nn.GRU(emb_dim + hid*2, hid, batch_first=True)
        self.out = nn.Linear(hid + hid*2 + emb_dim, vocab_size)
    def forward(self, y, hidden, enc_out, mask):
        # y: [B,1]; hidden: [1,B,H]
        e = self.emb(y)                                          # [B,1,E]
        ctx, attn = self.attn(hidden.squeeze(0), enc_out, mask)  # [B,2H]
        gru_in = torch.cat([e.squeeze(1), ctx], dim=1).unsqueeze(1)  # [B,1,E+2H]
        out, hidden = self.gru(gru_in, hidden)                   # out:[B,1,H]
        logits = self.out(torch.cat([out.squeeze(1), ctx, e.squeeze(1)], dim=1))
        return logits, hidden, attn

class Seq2Seq(nn.Module):
    def __init__(self, src_vsize, tgt_vsize, emb=128, hid=256):
        super().__init__()
        self.enc = Encoder(src_vsize, emb, hid)
        self.dec = Decoder(tgt_vsize, emb, hid)
        self.tgt_vsize = tgt_vsize
    def forward(self, src, tgt, teacher_forcing=0.5):
        B, T = tgt.shape
        mask = (src != PAD)
        enc_out, hidden = self.enc(src)
        outputs = torch.zeros(B, T, self.tgt_vsize, device=src.device)
        y = tgt[:, 0:1]  # SOS
        for t in range(1, T):
            logits, hidden, _ = self.dec(y, hidden, enc_out, mask)
            outputs[:, t] = logits
            use_tf = random.random() < teacher_forcing
            y = tgt[:, t:t+1] if use_tf else logits.argmax(1, keepdim=True)
        return outputs
    @torch.no_grad()
    def translate(self, src, max_len=MAX_LEN):
        mask = (src != PAD)
        enc_out, hidden = self.enc(src)
        B = src.size(0)
        y = torch.full((B,1), SOS, dtype=torch.long, device=src.device)
        outputs = []
        for _ in range(max_len-1):
            logits, hidden, _ = self.dec(y, hidden, enc_out, mask)
            y = logits.argmax(1, keepdim=True)
            outputs.append(y)
        return torch.cat(outputs, dim=1)

def ids_to_text(ids, inv_vocab):
    out = []
    for i in ids:
        i = int(i)
        if i == EOS: break
        if i in (PAD, SOS): continue
        out.append(inv_vocab.get(i, '<unk>'))
    return ' '.join(out)

In [6]:
from tqdm.auto import tqdm

model_s2s = Seq2Seq(len(src_vocab), len(tgt_vocab)).to(DEVICE)
opt = torch.optim.Adam(model_s2s.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

EPOCHS = 8
t0 = time.time()
for epoch in range(EPOCHS):
    model_s2s.train(); total = 0; n = 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for src, tgt in pbar:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        opt.zero_grad()
        out = model_s2s(src, tgt, teacher_forcing=0.5)
        loss = loss_fn(out[:,1:].reshape(-1, len(tgt_vocab)), tgt[:,1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_s2s.parameters(), 1.0)
        opt.step()
        total += loss.item(); n += 1
        pbar.set_postfix({'loss': f'{total/n:.3f}'})
s2s_time = time.time() - t0
print(f'Seq2Seq train time: {s2s_time:.1f}s')

# Generate translations on test set
model_s2s.eval()
s2s_preds = []
for src, tgt in test_dl:
    src = src.to(DEVICE)
    out = model_s2s.translate(src)
    for row in out: s2s_preds.append(ids_to_text(row.cpu().tolist(), tgt_inv))
print('\n--- Sample Seq2Seq translation ---')
print('SRC :', data['test_en'][0])
print('REF :', data['test_de'][0])
print('PRED:', s2s_preds[0])

Epoch 1/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 2/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 3/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 4/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 5/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 6/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 7/8:   0%|          | 0/454 [00:00<?, ?it/s]

Epoch 8/8:   0%|          | 0/454 [00:00<?, ?it/s]

Seq2Seq train time: 395.6s

--- Sample Seq2Seq translation ---
SRC : A man in an orange hat starring at something.
REF : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
PRED: ein mann mit einem orangen hut auf etwas .


## 5. Model 2 — opus-mt-en-de (Pre-trained Transformer)

Helsinki-NLP's `opus-mt-en-de` is a Transformer-based model trained on the OPUS multilingual corpus. We use it for inference only.

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_NAME = 'Helsinki-NLP/opus-mt-en-de'
mt_tok = AutoTokenizer.from_pretrained(MT_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(DEVICE)
mt_model.eval()

def mt_translate(texts, batch_size=32):
    out_all = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = mt_tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            gen = mt_model.generate(**enc, max_length=128, num_beams=4, early_stopping=True)
        out_all.extend(mt_tok.batch_decode(gen, skip_special_tokens=True))
    return out_all

t0 = time.time()
transformer_preds = mt_translate(data['test_en'])
transformer_time = time.time() - t0
print(f'Transformer inference time: {transformer_time:.1f}s for {len(data["test_en"])} sentences')
print('\n--- Sample Transformer translation ---')
print('SRC :', data['test_en'][0])
print('REF :', data['test_de'][0])
print('PRED:', transformer_preds[0])

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Transformer inference time: 16.2s for 1000 sentences

--- Sample Transformer translation ---
SRC : A man in an orange hat starring at something.
REF : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
PRED: Ein Mann in einem orangenen Hut, mit etwas.


## 6. Evaluation: BLEU, METEOR, ChrF, BERTScore

In [8]:
import sacrebleu
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score_fn

refs = data['test_de']

def evaluate_mt(preds, refs, label):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score / 100.0
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score / 100.0
    meteor_scores = [meteor_score([word_tokenize(r)], word_tokenize(p)) for p, r in zip(preds, refs)]
    P, R, F1 = bert_score_fn(preds, refs, lang='de', verbose=False, device=DEVICE.type)
    return {
        'Model': label,
        'BLEU': bleu,
        'METEOR': np.mean(meteor_scores),
        'ChrF': chrf,
        'BERTScore-F1': F1.mean().item(),
    }

s2s_metrics = evaluate_mt(s2s_preds, refs, 'Seq2Seq + Attention')
tx_metrics  = evaluate_mt(transformer_preds, refs, 'Transformer (opus-mt)')
summary = pd.DataFrame([s2s_metrics, tx_metrics])
summary['Time (s)'] = [s2s_time, transformer_time]
print(summary.to_string(index=False))
summary.to_csv('results/q4_summary.csv', index=False)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                Model     BLEU   METEOR     ChrF  BERTScore-F1   Time (s)
  Seq2Seq + Attention 0.043800 0.549530 0.410231      0.760309 395.627324
Transformer (opus-mt) 0.363742 0.665525 0.642305      0.896960  16.193487


## 7. Qualitative Examples

In [9]:
examples = []
for i in [0, 1, 2]:
    ex = {
        'src': data['test_en'][i],
        'reference': refs[i],
        'seq2seq': s2s_preds[i],
        'transformer': transformer_preds[i],
    }
    examples.append(ex)
    print(f'\n========== EXAMPLE {i+1} ==========')
    print(f"SRC        : {ex['src']}")
    print(f"REFERENCE  : {ex['reference']}")
    print(f"SEQ2SEQ    : {ex['seq2seq']}")
    print(f"TRANSFORMER: {ex['transformer']}")

with open('results/q4_examples.json','w') as f: json.dump(examples, f, indent=2, ensure_ascii=False)
with open('results/q4_full.json','w') as f:
    json.dump({'seq2seq': s2s_metrics, 'transformer': tx_metrics,
               's2s_time': s2s_time, 'tx_time': transformer_time}, f, indent=2)
print('\nResults saved to results/')


========== EXAMPLE 1 ==========
SRC        : A man in an orange hat starring at something.
REFERENCE  : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
SEQ2SEQ    : ein mann mit einem orangen hut auf etwas .
TRANSFORMER: Ein Mann in einem orangenen Hut, mit etwas.

========== EXAMPLE 2 ==========
SRC        : A Boston Terrier is running on lush green grass in front of a white fence.
REFERENCE  : Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
SEQ2SEQ    : ein <unk> läuft auf dem <unk> vor einem weißen zaun .
TRANSFORMER: Ein Boston Terrier läuft auf üppig grünem Gras vor einem weißen Zaun.

========== EXAMPLE 3 ==========
SRC        : A girl in karate uniform breaking a stick with a front kick.
REFERENCE  : Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
SEQ2SEQ    : ein mädchen gekleidetes mädchen macht einen einen mit mit einem fußtritt tritt .
TRANSFORMER: Ein Mädchen in Karate-Uniform bricht einen Stock mit einem Front-Kick.

R

## 8. Discussion (in Report)

See `reports/CENG467_Midterm_Report.pdf` for analysis of:
- BLEU vs ChrF behavior on morphologically rich German
- METEOR's role in synonym/stem matching
- BERTScore's semantic similarity perspective
- Architecture trade-offs: small Seq2Seq trained from scratch vs large pre-trained Transformer
- Handling of rare words and long-range dependencies